# 🚀 StartupIQ - Phase 1: Data Understanding

**Author:** Senior Data Analyst  
**Company:** Fortune 500  
**Project:** StartupIQ Portfolio Project  
**Date:** July 2026  

---

## Executive Summary
This notebook performs an initial exploratory and descriptive analysis of the raw startup dataset. The goal is to gain an understanding of the structure, features, scale, and potential quality anomalies of the data before passing it to downstream preprocessing and databases. 

*Note: In accordance with the project guidelines, no modifications or cleaning actions are performed in this notebook. It is strictly for data inspection.*

## 1. Import Libraries
We import standard data manipulation, visualization, and OS/path utilities.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import pathlib
import sys

print("Libraries imported successfully.")
print(f"Python version: {sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 2. Load Dataset
We resolve paths dynamically using `pathlib` to scan the `data/raw/` directory and load the raw CSV dataset.

In [ ]:
# Resolve paths relative to the notebooks directory
current_dir = pathlib.Path.cwd()
raw_dir = current_dir.parent / "data" / "raw" if (current_dir.parent / "data" / "raw").exists() else current_dir / "data" / "raw"

csv_files = list(raw_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in directory: {raw_dir.resolve()}")

raw_file_path = csv_files[0]
print(f"Found raw file: {raw_file_path.name}")
print(f"Full path: {raw_file_path.resolve()}")

# Load raw dataset
df = pd.read_csv(raw_file_path)
print(f"Dataset successfully loaded. Records count: {len(df):,}")

## 3. Dataset Overview
We analyze the basic structural properties of the dataset: shape, column types, count, and memory footprint.

In [ ]:
shape = df.shape
num_rows, num_cols = shape
col_names = df.columns.tolist()
dtypes = df.dtypes
memory_usage_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)

print("=== DATASET STRUCTURE ===")
print(f"Shape of Dataset: {shape}")
print(f"Number of Rows: {num_rows:,}")
print(f"Number of Columns: {num_cols}")
print(f"Memory Footprint: {memory_usage_mb:.2f} MB\n")

print("=== COLUMN NAMES & TYPES ===")
for col in col_names:
    print(f"- {col:<26} | Type: {str(dtypes[col]):<8} | Non-Null Count: {df[col].count():,}")

## 4. Preview Data
Previewing the head, tail, and random samples of the dataset to verify formatting and row consistency.

In [ ]:
print("=== FIRST 10 ROWS ===")
display(df.head(10))

print("\n=== LAST 10 ROWS ===")
display(df.tail(10))

print("\n=== RANDOM 10 ROWS ===")
display(df.sample(10, random_state=42))

## 5. Missing Value Analysis
Identify any missing/null entries across columns to determine potential imputation tasks.

In [ ]:
missing_count = df.isnull().sum()
missing_percentage = (missing_count / len(df)) * 100

missing_analysis = pd.DataFrame({
    'Column Name': missing_count.index,
    'Missing Count': missing_count.values,
    'Missing Percentage (%)': missing_percentage.values
}).sort_values(by='Missing Percentage (%)', ascending=False).reset_index(drop=True)

print("=== MISSING VALUE SUMMARY ===")
display(missing_analysis)

## 6. Duplicate Analysis
Verify if there are redundant row records that should be deduplicated during cleaning.

In [ ]:
dup_count = df.duplicated().sum()
dup_percentage = (dup_count / len(df)) * 100

print("=== DUPLICATE ANALYSIS ===")
print(f"Total Rows Checked: {len(df):,}")
print(f"Duplicate Rows Count: {dup_count:,}")
print(f"Duplicate Ratio: {dup_percentage:.4f}%")

## 7. Numerical Summary
Descriptive statistics for numerical columns are generated below.

In [ ]:
num_summary = df.describe().T
display(num_summary)

### Detailed Interpretation of Numerical Metrics

1. **`funding_rounds`**:
   - **Count**: 100,000 valid records (no missing values).
   - **Mean / Median (50%)**: Average is ~2.00 rounds, and the median is 2. This suggests a centered distribution where most startups raise between 1 and 3 rounds.
   - **Spread (Min/Max)**: Ranges from 0 (bootstrapped startups) to 8 rounds (mature startups). Standard deviation is ~1.41.

2. **`founder_experience_years`**:
   - **Mean / Median (50%)**: Average is ~12.02 years, median is 12 years. This shows that founding teams have substantial career experience on average.
   - **Spread (Min/Max)**: Ranges from 0 (first-time founders right after university/industry entry) to 24 years (seasoned industry veterans).

3. **`team_size`**:
   - **Mean / Median (50%)**: Average is 150.73 employees, median is 151.
   - **Spread (Min/Max)**: Ranges from 2 (early-stage core team) to 299 employees. The uniform-like dispersion suggests this dataset covers startups across all growth phases.

4. **`market_size_billion`**:
   - **Mean / Median (50%)**: Average is $33.20B, median is $20.16B.
   - **Outliers & Spread**: Max is $1,072.43B, while the 75th percentile is $39.53B. The massive difference between the median/75th percentile and the maximum suggests strong positive skewness with outlier markets.

5. **`product_traction_users`**:
   - **Mean / Median (50%)**: Average is 285,422.8 users, median is 264,989.
   - **Spread (Min/Max)**: Ranges from 668 to 915,203 users, reflecting wide variation in adoption and user base scaling.

6. **`burn_rate_million`**:
   - **Mean / Median (50%)**: Average burn rate is 16.78, median is 12.17.
   - **Outliers**: Max is 357.49, indicating extreme cases where capital is burned at an exceptionally rapid pace.

7. **`revenue_million`**:
   - **Scale Anomaly**: The mean is $7.828191 \times 10^5$ (782,819.1 USD), and the maximum is $4,168,443.0$ (4.17M USD).
   - **Note**: Although the column name contains the suffix `_million`, the values are recorded in raw USD units. If they were in millions, it would represent trillions of dollars, which is not applicable for startups. This is a critical data quality issue to address in data cleaning.

## 8. Categorical Summary
We analyze categorical variables to identify cardinality (unique values count), mode (top value), and its respective frequency.

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns
cat_summary = []

for col in cat_cols:
    val_counts = df[col].value_counts()
    unique_vals = df[col].nunique()
    top_val = val_counts.index[0]
    freq = val_counts.iloc[0]
    pct = (freq / len(df)) * 100
    
    cat_summary.append({
        'Categorical Column': col,
        'Unique Count': unique_vals,
        'Top Value': top_val,
        'Frequency': freq,
        'Percentage (%)': round(pct, 2)
    })

cat_summary_df = pd.DataFrame(cat_summary)
display(cat_summary_df)

print("=== CATEGORICAL DISTRIBUTIONS ===")
for col in cat_cols:
    print(f"\nDistribution for column '{col}':")
    print(df[col].value_counts(normalize=True) * 100)

## 9. Business Understanding
We classify columns into thematic categories matching standard startup metrics categories (e.g. founder, market, financials, outcomes).

In [ ]:
business_categories = {
    'Company Information': ['team_size', 'sector'],
    'Founder Information': ['founder_experience_years', 'founder_background'],
    'Funding Information': ['funding_rounds', 'investor_type'],
    'Financial Metrics': ['burn_rate_million', 'revenue_million'],
    'Startup Outcome (Target)': ['outcome'],
    'Market Information': ['market_size_billion'],
    'Product Information / Traction': ['product_traction_users'],
    'Geographic Information': ['None present in raw data'],
    'Time Information': ['None present in raw data']
}

print("=== STAKEHOLDER BUSINESS DOMAINS ===")
for domain, cols in business_categories.items():
    print(f"\n📂 {domain}:")
    for col in cols:
        print(f"  - {col}")

## 10. Initial Observations & Action Plan

Based on the exploratory overview, we identify the following key observations:

### 🔍 Key Findings

1. **Missing Values**:
   - **No Missing Values**: Every feature is populated for all 100,000 startup profiles. No basic imputation (mean/median/mode filling) is required.

2. **Duplicate Records**:
   - **0 duplicates detected**: The dataset has unique combinations of numeric values, signifying clean row representations.

3. **Data Quality & Naming Inconsistencies**:
   - **`revenue_million` Scaling Issue**: The values range from ~1,344 USD to ~4,168,443 USD. Since the average is $782.8$k, it represents single USD units rather than millions (unlike the column name). 
   - **`burn_rate_million` Scaling**: The values range from ~0.28 to ~357.49, which are in millions (e.g. mean is 16.78M USD).
   - **Cleaning Needed**: We must align these scales during the data cleaning phase (either scaling `revenue_million` down by dividing by 1,000,000 or scaling `burn_rate_million` up by multiplying by 1,000,000 to keep units matching).

4. **Outliers**:
   - **`market_size_billion`**: The max value is $1,072.4B (over 1 trillion USD addressable market). Standard scaling or box-cox transformation might be needed to handle extreme positive skewness.
   - **`burn_rate_million`**: Max is $357.49M, significantly higher than the 75th percentile ($20.95M).

5. **Baseline Categorical Representation**:
   - **`investor_type`**: Contains a large base of `none` (24,866 entries), indicating bootstrapped setups. This is a vital baseline classification marker.

---

### 🚀 Next Cleaning & Preprocessing Steps
- **Unit Scaling**: Align revenue and burn rate columns to absolute USD or million USD equivalents to calculate cash runways.
- **Feature Transformations**: Address skewness in `market_size_billion` and `burn_rate_million` using robust scalers.
- **Variable Encoding**: One-hot encode `sector`, `founder_background`, and `investor_type`.
- **Binary Target Mapping**: Group outcomes for classification (e.g., Success [IPO / Acquisition] vs Failure).